In [1]:
##### Converts raw rasters on GDP and population into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # GDP (USD PPP) per person

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# rasters
GDP_pc = f"{cd}/Data/Raw/Predictors/GDP_Kummu/rast_adm2_gdp_perCapita_1990_2022.tif"

# save paths
pixels = f"{cd}/Data/Clean/Predictors/Rasters/GDP_per_capita.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/GDP_per_capita.csv"

In [6]:
#### Run resampling for pixel scale

### PREP

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

### COPY WITH REFERENCE METADATA
with rasterio.open(GDP_pc) as src:
    data      = src.read(31).astype(np.float32)
    src_nodata = src.nodata

# replace any nodata with -9999
if src_nodata is not None:
    data[data == src_nodata] = -9999
data[np.isnan(data)] = -9999

out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

with rasterio.open(pixels, "w", **out_meta) as dst:
    dst.write(data, 1)

In [7]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
with rasterio.open(pixels) as src:
    raster_crs = src.crs

gdf_proj = total_geo.to_crs(raster_crs)

# set final form 
result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE
for col_name, rpath in [("GDP_pc", pixels)]:
    zonal = rasterstats.zonal_stats(
        gdf_proj,
        rpath,
        stats="mean",
        nodata=-9999
    )
    result[col_name] = [z["mean"] for z in zonal]

### SAVE
result.to_csv(vectors, index=False)